# Activité Pratique N°4 — Architecture Transformer
## Partie 1 — Étape 5 : Modèle de type GPT

**IA Générative Multi Modale — 5IAD G1, EMSI** · Étudiant : BENHAYOUN SADAFI Mohamed

Créer et tester un exemple basé sur un Transformer de type GPT : prédire la suite d'un texte.

GPT-2 est un **décodeur Transformer à attention causale** : chaque token ne voit que les tokens précédents.
Il génère **un token à la fois** : il calcule les probabilités du token suivant, en choisit un, l'ajoute à la séquence et recommence.

```
Input : séquence « Artificial Intelligence can… » ──► Generative Pre-Trained Transformer ──► Output : « …help people and organizations… »
```

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
modele = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
tokenizer.pad_token = tokenizer.eos_token          # GPT-2 n'a pas de token de remplissage
modele.config.pad_token_id = tokenizer.pad_token_id
print("Paramètres :", sum(p.numel() for p in modele.parameters()) / 1e6, "millions")

### 1. Tokenisation et prédiction d'un seul token suivant

In [ ]:
amorce = "Artificial intelligence can"
entrees = tokenizer(amorce, return_tensors="pt").to(device)
print("Tokens :", tokenizer.convert_ids_to_tokens(entrees["input_ids"][0]))
print("Dimensions de l'entrée :", tuple(entrees["input_ids"].shape))

with torch.no_grad():
    logits = modele(**entrees).logits          # (1, nb_tokens, taille_vocabulaire)
print("Logits :", tuple(logits.shape))

probas = torch.softmax(logits[0, -1], dim=-1)  # distribution du token suivant
top = torch.topk(probas, 5)
print(f"\nToken suivant le plus probable après {amorce!r} :")
for p, i in zip(top.values, top.indices):
    print(f"  {p.item():6.2%}  {tokenizer.decode(i)!r}")

### 2. Générer la suite du texte, token après token

In [ ]:
torch.manual_seed(0)
with torch.no_grad():
    sorties = modele.generate(**entrees, max_new_tokens=50, do_sample=True, temperature=0.8, top_p=0.9,
                              num_return_sequences=3, pad_token_id=tokenizer.pad_token_id)

for k, sequence in enumerate(sorties, 1):
    print(f"--- Suite {k} ---")
    print(tokenizer.decode(sequence, skip_special_tokens=True), "\n")

- `do_sample=True` : on tire le token au hasard selon les probabilités (sinon : toujours le plus probable, texte répétitif).
- `temperature=0.8` : distribution un peu plus pointue que l'originale.
- `top_p=0.9` : on ne garde que les tokens les plus probables dont la somme des probabilités atteint 90 %.

**À retenir :** GPT prédit le prochain token à partir du passé. Il produit une suite **plausible**, sans garantie qu'elle soit vraie.